## Sélection des variables : base logement et LOVAC

### Objectif
Garder seulement les variables utiles pour prédire le prix au m², puis livrer au groupe deux fichiers allégés, sans les prix, pour le merge global avec les autres sources.

### Démarche en 3 temps

| Partie | Action | Fichier créé ? |
|---|---|---|
| 1. Merge temporaire | Table cible + base logement + LOVAC | Non (en mémoire) |
| 2. Sélection | Séparation train (2014-2024) / test (2025), puis analyse **sur train uniquement** | Non (en mémoire) |
| 3. Fichiers allégés | On repart des fichiers d'origine et on retire les variables écartées | Oui : 2 nouveaux fichiers |

### Le merge temporaire
Il sert uniquement à mettre les prix à côté de nos variables pour les évaluer. La table cible sert de base : on lui ajoute des colonnes, sans retirer ni dupliquer de ligne (`how="left"`).

| Étape | Clé de jointure | Point d'attention |
|---|---|---|
| Harmonisation des codes | Table de passage INSEE (ancien code → code 2026) | Les 3 fichiers n'ont pas la même version des codes (cible : année de la vente, logement : 2024, LOVAC : 2026). Sans cette étape, 5 108 lignes restaient sans données ; il en reste 63. |
| Ajout de la base logement | Commune (code d'origine, sinon code 2026) | Le recensement 2022 est recopié pour toutes les années (`rp_anticipe` pour les ventes avant 2022). |
| Ajout de LOVAC | Commune (code 2026) + année | Une vente de l'année N reçoit le millésime N (situation au 1er janvier N-1). Pas de données LOVAC avant 2020. |

**Contrôles :** 367 779 lignes après chaque jointure (identique à la table cible) et une seule ligne par commune × année.

**Limite :** une commune fusionnée reçoit les caractéristiques de la nouvelle commune entière (signalée par `commune_fusionnee`).

### Méthode de sélection
- **Cibles** : log du prix médian au m², maisons et appartements séparément, seulement si la commune a au moins 5 ventes dans l'année (une médiane sur 1 ou 2 ventes n'est pas fiable). Cela donne 195 568 lignes pour les maisons et 35 958 pour les appartements.
- **Lien avec le prix** : corrélation de Spearman, choisie parce que la relation n'est pas forcément linéaire et que ce coefficient est peu sensible aux valeurs extrêmes.
- **Vérification** : même calcul sur 2024 seule (une ligne par commune). Les résultats sont presque identiques : la répétition des communes sur plusieurs années ne fausse pas les coefficients.
- **Règle** : on garde une variable si |ρ| > 0,2 pour les maisons ou pour les appartements.
- **Doublons** : si deux variables sont corrélées à plus de 0,8 entre elles, on n'en garde qu'une.

### Résultats (train 2014-2024)

| Variable | Source | ρ maisons | ρ apparts | % manquants | Décision |
|---|---|---|---|---|---|
| taux_vacance_2ans | LOVAC | -0,69 | -0,59 | 85,4 | À tester |
| part_longue_parmi_vacants | LOVAC | -0,68 | -0,60 | 82,1 | Écartée (doublon de taux_vacance_2ans, 0,88) |
| part_avant1946 | Recensement | -0,61 | -0,35 | 0 | Gardée |
| taux_vacance | Recensement | -0,50 | -0,48 | 0 | Gardée |
| taux_vacance_complete | LOVAC | -0,46 | -0,34 | 54,5 | Gardée |
| anciennete_moyenne | Recensement | -0,45 | -0,15 | 0 | Gardée |
| part_2006_2019 | Recensement | 0,42 | 0,30 | 0 | Gardée |
| part_apparts | Recensement | 0,40 | 0,14 | 0 | Gardée |
| part_maisons | Recensement | -0,39 | -0,14 | 0 | Écartée (doublon de part_apparts, -0,96) |
| log_nb_logements | Recensement | 0,38 | 0,10 | 0 | Gardée |
| part_parking | Recensement | 0,24 | 0,32 | 0 | Gardée |
| taille_menage | Recensement | 0,30 | 0,24 | 0 | Gardée |
| part_rsec | Recensement | -0,25 | 0,16 | 0 | Gardée |
| part_suroccupes | Recensement | 0,24 | 0,23 | 0 | Gardée |
| part_sans_voiture | Recensement | -0,17 | -0,23 | 0 | Gardée |
| part_moins40m2 | Recensement | 0,20 | 0,21 | 0 | Gardée |
| part_locataires | Recensement | 0,16 | -0,20 | 0 | Écartée (lien faible) |
| part_hlm | Recensement | 0,17 | -0,16 | 0 | Écartée (lien faible) |
| part_emmenages_2ans | Recensement | 0,17 | -0,13 | 0 | Écartée (lien faible) |
| part_proprio | Recensement | -0,15 | 0,17 | 0 | Écartée (lien faible, doublon de part_locataires) |
| part_2voitures | Recensement | 0,16 | 0,17 | 0 | Écartée (lien faible) |
| nb_pieces_moyen | Recensement | -0,12 | -0,07 | 0 | Écartée (lien faible) |
| part_100m2_plus | Recensement | 0,01 | 0,10 | 0 | Écartée (lien faible) |

Les 54,5 % de manquants de `taux_vacance_complete` correspondent aux années 2014 à 2019, où LOVAC n'existe pas. La variable est complète de 2020 à 2025.

**Bilan : 12 variables du recensement gardées, 1 de LOVAC gardée, 1 de LOVAC à tester, 9 écartées.**

### Variables retenues

Au final, **14 variables** sont retenues : 12 issues du recensement et 2 issues de LOVAC.

**Recensement** (fichier `logement_commune_selection.csv`)

| Variable | Ce qu'elle mesure |
|---|---|
| `part_avant1946` | Part des résidences principales construites avant 1946 |
| `taux_vacance` | Part des logements vacants |
| `anciennete_moyenne` | Nombre moyen d'années depuis l'emménagement des ménages |
| `part_2006_2019` | Part des résidences principales construites entre 2006 et 2019 |
| `part_apparts` | Part d'appartements |
| `log_nb_logements` | Taille de la commune (nombre de logements, en log) |
| `part_parking` | Part des ménages avec une place de stationnement |
| `taille_menage` | Nombre moyen de personnes par ménage |
| `part_suroccupes` | Part des résidences principales suroccupées |
| `part_rsec` | Part de résidences secondaires |
| `part_sans_voiture` | Part des ménages sans voiture |
| `part_moins40m2` | Part des résidences principales de moins de 40 m² |

**LOVAC** (fichier `lovac_commune_annee_selection.csv`)

| Variable | Ce qu'elle mesure | Statut |
|---|---|---|
| `taux_vacance_complete` | Taux de vacance du parc privé (complété par le taux de l'EPCI si caché) | Gardée |
| `taux_vacance_2ans` | Taux de logements vacants depuis plus de 2 ans | À tester (souvent manquante) |

**Autres colonnes des fichiers (pas des variables pour le modèle)**
- **Identifiants** pour le merge global : `COM`, `COM_ENTIERE`, `EPCI`, `dep`, `millesime`, `annee_vente_associee`.
- **Signalements** : `petite_commune`, `taux_impute`, `apres_rupture`, `alsace_moselle`, `mayotte`.
- **Effectifs bruts** (`P22_...`, `vacants`, `parc_prive`) : ils servent seulement à recalculer les taux si le groupe regroupe les communes (ex. bassin de vie).


### Fichiers livrés au groupe

| Fichier | Niveau | Lignes | Colonnes | Contenu |
|---|---|---|---|---|
| `logement_commune_selection.csv` | Commune (ou arrondissement) | 34 955 | 36 | Identifiants, 12 indicateurs gardés, effectifs bruts correspondants, signalements |
| `lovac_commune_annee_selection.csv` | Commune × millésime (2020-2026) | 244 384 | 16 | Identifiants, années, vacants, parc privé, 2 taux gardés, signalements |

- Les deux fichiers couvrent **toutes les communes et toutes les années, 2025 comprise**. La séparation train / test se fera dans le merge global.
- Les **effectifs bruts** sont conservés dans le fichier logement pour pouvoir recalculer les taux si le groupe regroupe les communes (ex. bassin de vie) : on additionne les effectifs, on ne fait jamais la moyenne des taux.

### Constat
Les variables les plus liées au prix décrivent l'**attractivité du marché local** :
- un parc ancien, une vacance élevée et des habitants installés depuis longtemps vont avec des prix plus bas ;
- des constructions récentes et des communes plus grandes vont avec des prix plus élevés.

Les liens sont plus forts pour les maisons que pour les appartements.

### Impact projet
- Les variables LOVAC sont parmi les plus liées au prix, mais elles sont vides de 2014 à 2019 et complètes en 2025 : l'entraînement et le test n'ont pas la même couverture LOVAC.
- `taux_vacance_2ans` n'est connue que pour environ un tiers des communes, surtout les plus grandes (secret statistique). Son lien fort est mesuré sur un sous-ensemble qui n'est pas représentatif.

### Action envisagée
- Merge global du groupe : table cible + table de passage, puis nos deux fichiers et les autres sources (mêmes clés que ci-dessus), puis séparation train / test et preprocessing final (seuil de ventes, log du prix, valeurs manquantes calculées sur train).
- Décider en groupe pour LOVAC :
  - limiter l'entraînement à 2020-2024 ;
  - ou garder 2014-2024 avec un modèle qui gère les valeurs manquantes ;
  - ou retirer LOVAC.
- Comparer les performances du modèle avec et sans `taux_vacance_2ans`.
- Refaire un contrôle des doublons sur la table globale : une variable d'une autre source peut répéter l'information d'une des nôtres.

### Incertitudes
- Spearman ne détecte que les liens qui vont toujours dans le même sens. Une variable écartée pourrait quand même être utile à un modèle à arbres ; à vérifier avec l'importance des variables.
- Le seuil de 0,2 est un choix : `part_locataires` (0,20) est écartée de justesse.
- `part_rsec` a un effet opposé pour les maisons (-) et les appartements (+), sans doute à cause des stations touristiques où les appartements sont chers. À interpréter avec prudence.
- Le recensement 2022 est utilisé pour toutes les années, y compris pour les ventes de 2014 à 2021.

In [3]:
import numpy as np
import pandas as pd
from scipy import stats

# ═══════════════════════════════════════════════════════════════════════════
# ÉTAPE 0 : chemins
# ═══════════════════════════════════════════════════════════════════════════
# Fichiers lus (jamais modifiés)
chemin_cible    = "../data/dataset_cible/cible_commune_annee_type.csv"
chemin_logement = "../data/processed/logement_commune_processed_2022.csv"
chemin_lovac    = "../data/processed/lovac_commune_annee.csv"
chemin_passage  = "../data/table_passage_geo2003_geo2026.xlsx"

# Seuls fichiers créés : les deux sources allégées
chemin_logement_final = "../data/processed/logement_commune_selection.csv"
chemin_lovac_final    = "../data/processed/lovac_commune_annee_selection.csv"


# ═══════════════════════════════════════════════════════════════════════════
# PARTIE 1 : MERGE TEMPORAIRE (en mémoire, aucun fichier créé)
# Il sert uniquement à mettre les prix à côté de nos variables pour les évaluer.
# ═══════════════════════════════════════════════════════════════════════════

# ── 1.1 Chargement (codes commune en texte) ──────────────────────────────────
cible = pd.read_csv(chemin_cible, dtype={"code_commune": str})
logement = pd.read_csv(chemin_logement, sep=";", dtype={"COM": str, "COM_ENTIERE": str, "dep": str})
lovac = pd.read_csv(chemin_lovac, sep=";", dtype={"COM": str, "COM_ENTIERE": str, "dep": str, "EPCI": str})
print("Cible :", cible.shape, "| Logement :", logement.shape, "| LOVAC :", lovac.shape)

# ── 1.2 Codes des communes fusionnées -> code 2026 ───────────────────────────
passage = pd.read_excel(chemin_passage, sheet_name="Table de passage", skiprows=5, dtype=str)
passage = passage[["CODGEO_INI", "CODGEO_2026"]].drop_duplicates(subset="CODGEO_INI")
passage.columns = ["code_commune", "code_2026"]
table = cible.merge(passage, on="code_commune", how="left")
table["code_2026"] = table["code_2026"].fillna(table["code_commune"])   # arrondissements

# ── 1.3 Ajout de la base logement (géographie 2024) ──────────────────────────
codes_logement = set(logement["COM"])
table["code_logement"] = np.where(table["code_commune"].isin(codes_logement),
                                  table["code_commune"], table["code_2026"])
table = table.merge(logement.drop(columns=["dep", "alsace_moselle"]),
                    left_on="code_logement", right_on="COM", how="left").drop(columns=["COM"])

# ── 1.4 Ajout de LOVAC (géographie 2026, vente de l'année N -> millésime N) ──
colonnes_lovac = ["COM", "annee_vente_associee", "taux_vacance_complete",
                  "taux_vacance_2ans", "part_longue_parmi_vacants"]
table = table.merge(lovac[colonnes_lovac],
                    left_on=["code_2026", "annee"], right_on=["COM", "annee_vente_associee"],
                    how="left").drop(columns=["COM", "annee_vente_associee"])
print("Table fusionnée :", table.shape, "(doit avoir", len(cible), "lignes)")

# ── 1.5 Séparation train / test (en mémoire) ─────────────────────────────────
# 2025 est réservée au test : on ne la regarde PAS pour choisir les variables.
train = table[table["annee"] <= 2024].copy()
print("Train (2014-2024) :", train.shape)


# ═══════════════════════════════════════════════════════════════════════════
# PARTIE 2 : SÉLECTION DES VARIABLES (sur TRAIN uniquement)
# ═══════════════════════════════════════════════════════════════════════════

# ── 2.1 Cibles : log du prix, seulement si au moins 5 ventes (médiane fiable) ─
SEUIL_VENTES = 5
train["log_prix_maison"] = np.where(train["n_maison"] >= SEUIL_VENTES, np.log(train["prix_m2_med_maison"]), np.nan)
train["log_prix_appart"] = np.where(train["n_appart"] >= SEUIL_VENTES, np.log(train["prix_m2_med_appart"]), np.nan)

recensement = ["taux_vacance", "part_rsec", "part_maisons", "part_apparts", "part_moins40m2",
               "part_100m2_plus", "nb_pieces_moyen", "part_avant1946", "part_2006_2019",
               "part_proprio", "part_locataires", "part_hlm", "part_emmenages_2ans",
               "anciennete_moyenne", "taille_menage", "part_sans_voiture", "part_2voitures",
               "part_parking", "part_suroccupes", "log_nb_logements"]
variables_lovac = ["taux_vacance_complete", "taux_vacance_2ans", "part_longue_parmi_vacants"]
candidates = recensement + variables_lovac

# ── 2.2 Lien avec le prix : Spearman (tout le train, puis 2024 seule) ────────
# 2024 seule = une ligne par commune : vérifie que la répétition des communes
# sur plusieurs années ne change pas le résultat.
train_2024 = train[train["annee"] == 2024]

def spearman(df, x, y):
    ok = df[x].notna() & df[y].notna()
    return round(stats.spearmanr(df.loc[ok, x], df.loc[ok, y])[0], 2)

lignes = []
for col in candidates:
    ligne = {"variable": col}
    ligne["source"] = "recensement" if col in recensement else "LOVAC"
    ligne["rho_maisons"] = spearman(train, col, "log_prix_maison")
    ligne["rho_apparts"] = spearman(train, col, "log_prix_appart")
    ligne["rho_maisons_2024"] = spearman(train_2024, col, "log_prix_maison")
    ligne["rho_apparts_2024"] = spearman(train_2024, col, "log_prix_appart")
    ligne["pct_manquants"] = round(train[col].isna().mean() * 100, 1)
    lignes.append(ligne)

tri = pd.DataFrame(lignes).set_index("variable")
tri["force"] = tri[["rho_maisons", "rho_apparts"]].abs().max(axis=1)
display(tri.sort_values("force", ascending=False))

# ── 2.3 Doublons : paires de variables corrélées à plus de 0,8 ───────────────
matrice = train[candidates].corr(method="spearman")
print("Paires en doublon (|rho| > 0,8) :")
for i in range(len(candidates)):
    for j in range(i + 1, len(candidates)):
        a = candidates[i]
        b = candidates[j]
        if abs(matrice.loc[a, b]) > 0.8:
            print("  ", a, "/", b, ":", round(matrice.loc[a, b], 2))

# ── 2.4 Choix (écrit à la main d'après les deux résultats ci-dessus) ─────────
# Règle : |rho| > 0,2 pour les maisons OU les appartements, une seule variable par doublon
indicateurs_gardes = [
    "part_avant1946", "taux_vacance", "anciennete_moyenne", "part_2006_2019",
    "part_apparts",          # à la place de part_maisons (doublon -0,96)
    "log_nb_logements", "part_parking", "taille_menage", "part_suroccupes",
    "part_rsec", "part_sans_voiture", "part_moins40m2",
]
lovac_gardees = [
    "taux_vacance_complete", # complète de 2020 à 2025 grâce au taux de l'EPCI
    "taux_vacance_2ans",     # à tester : lien fort mais souvent cachée (secret statistique)
]                            # part_longue_parmi_vacants écartée (doublon 0,88)

ecartees = []
for col in candidates:
    if col not in indicateurs_gardes and col not in lovac_gardees:
        ecartees.append(col)
print("Écartées :", ecartees)


# ═══════════════════════════════════════════════════════════════════════════
# PARTIE 3 : CRÉATION DES DEUX FICHIERS ALLÉGÉS (sans les prix)
# On repart des fichiers d'origine : toutes les communes et toutes les années,
# 2025 comprise. La séparation train/test se refera dans le merge global du groupe.
# ═══════════════════════════════════════════════════════════════════════════

# ── 3.1 Base logement ────────────────────────────────────────────────────────
# On garde aussi les effectifs bruts des indicateurs retenus : ils permettront de
# recalculer les taux si le groupe regroupe les communes (ex. bassin de vie).
effectifs_gardes = ["P22_LOG", "P22_RP", "P22_MEN",
                    "P22_LOGVAC", "P22_RSECOCC", "P22_APPART",            # vacance, rés. secondaires, apparts
                    "P22_RP_M30M2", "P22_RP_3040M2",                      # moins de 40 m²
                    "P22_RP_ACH1919", "P22_RP_ACH1945", "P22_RP_ACH2019", # âge du parc
                    "P22_ANEM_RP", "P22_NPER_RP",                         # ancienneté, taille ménage
                    "P22_RP_VOIT1P", "P22_RP_GARL",                       # voiture, parking
                    "C22_RP_SUROCC_MOD", "C22_RP_SUROCC_ACC"]             # suroccupation

colonnes_logement = (["COM", "COM_ENTIERE", "dep", "annee_rp", "nb_iris"]
                     + effectifs_gardes + indicateurs_gardes
                     + ["alsace_moselle", "petite_commune"])
logement_final = logement[colonnes_logement]

# ── 3.2 LOVAC ────────────────────────────────────────────────────────────────
colonnes_lovac_final = (["COM", "COM_ENTIERE", "EPCI", "dep",
                         "millesime", "annee_situation", "annee_vente_associee",
                         "vacants", "vacants_2ans", "parc_prive"]
                        + lovac_gardees
                        + ["taux_impute", "apres_rupture", "alsace_moselle", "mayotte"])
lovac_final = lovac[colonnes_lovac_final]

# ── 3.3 Contrôles ────────────────────────────────────────────────────────────
print("Logement : une ligne par commune :", logement_final["COM"].is_unique, "|", logement_final.shape)
print("LOVAC : une ligne par commune × millésime :",
      not lovac_final.duplicated(subset=["COM", "millesime"]).any(), "|", lovac_final.shape)

# ── 3.4 Enregistrement dans de NOUVEAUX fichiers ─────────────────────────────
logement_final.to_csv(chemin_logement_final, sep=";", index=False)
lovac_final.to_csv(chemin_lovac_final, sep=";", index=False)
print("Enregistré :", chemin_logement_final, "et", chemin_lovac_final)

Cible : (367779, 6) | Logement : (34955, 53) | LOVAC : (244384, 19)
Table fusionnée : (367779, 61) (doit avoir 367779 lignes)
Train (2014-2024) : (337503, 61)


,source,rho_maisons,rho_apparts,rho_maisons_2024,rho_apparts_2024,pct_manquants,force
variable,,,,,,,
taux_vacance_2ans,LOVAC,-0.69,-0.59,-0.67,-0.54,85.4,0.69
part_longue_parmi_vacants,LOVAC,-0.68,-0.60,-0.67,-0.53,82.1,0.68
part_avant1946,recensement,-0.61,-0.35,-0.65,-0.37,0.0,0.61
taux_vacance,recensement,-0.50,-0.48,-0.53,-0.53,0.0,0.50
taux_vacance_complete,LOVAC,-0.46,-0.34,-0.47,-0.38,54.5,0.46
anciennete_moyenne,recensement,-0.45,-0.15,-0.49,-0.17,0.0,0.45
part_2006_2019,recensement,0.42,0.30,0.47,0.35,0.0,0.42
part_apparts,recensement,0.40,0.14,0.42,0.16,0.0,0.40
part_maisons,recensement,-0.39,-0.14,-0.42,-0.16,0.0,0.39


Paires en doublon (|rho| > 0,8) :
   part_maisons / part_apparts : -0.96
   part_100m2_plus / nb_pieces_moyen : 0.81
   part_proprio / part_locataires : -0.97
   taux_vacance_complete / taux_vacance_2ans : 0.8
   taux_vacance_2ans / part_longue_parmi_vacants : 0.88
Écartées : ['part_maisons', 'part_100m2_plus', 'nb_pieces_moyen', 'part_proprio', 'part_locataires', 'part_hlm', 'part_emmenages_2ans', 'part_2voitures', 'part_longue_parmi_vacants']
Logement : une ligne par commune : True | (34955, 36)
LOVAC : une ligne par commune × millésime : True | (244384, 16)
Enregistré : ../data/processed/logement_commune_selection.csv et ../data/processed/lovac_commune_annee_selection.csv


### ANNEXE


### Le merge temporaire en détail

#### Pourquoi un merge ?
Pour savoir si une variable est utile, il faut la comparer au prix. Or nos variables (base logement, LOVAC) et les prix (table cible) sont dans des fichiers différents. Le merge les met côte à côte, sur la même ligne. Il est fait **en mémoire** : aucun fichier n'est créé, il sert seulement à l'analyse.

#### Les trois fichiers à assembler

| Fichier | Une ligne = | Années | Version des codes commune |
|---|---|---|---|
| Table cible (prix DVF+) | une commune × une année | 2014 à 2025 | Code de la commune **au moment de la vente** |
| Base logement (recensement) | une commune | 2022 seulement | Géographie **2024** |
| LOVAC | une commune × un millésime | 2020 à 2026 | Géographie **2026** |

La table cible sert de base : on lui **ajoute** des colonnes, sans jamais retirer ni dupliquer de ligne (`how="left"`).

#### Difficulté n°1 : les communes fusionnées
Chaque année, des communes fusionnent et prennent un nouveau code. Les trois fichiers n'utilisent donc pas la même version des codes.

*Exemple :* les communes A et B fusionnent en 2019 dans une « commune nouvelle » C.
- Dans la table cible, les ventes de 2014 à 2018 portent encore les codes de A et de B.
- Dans la base logement (2024) et dans LOVAC (2026), A et B n'existent plus : seul le code de C existe.
- Un merge direct ne trouverait donc rien pour les lignes de A et B.

**Solution : la table de passage de l'INSEE.** Elle donne, pour chaque ancien code, le code de la commune en 2026 (A → C, B → C). On ajoute ce code dans une colonne `code_2026`. Pour une commune qui n'a pas changé, `code_2026` est identique au code d'origine. Les arrondissements de Paris, Lyon et Marseille, absents de cette table, gardent leur code.

**Résultat :** sans cette étape, 5 108 lignes restaient sans données logement. Avec elle, il en reste 63, soit une vingtaine de communes dont le découpage a changé très récemment.

#### Difficulté n°2 : la base logement est en géographie 2024, LOVAC en 2026
Une commune qui a fusionné **en 2025 ou 2026** existe encore sous son ancien code dans la base logement, mais plus dans LOVAC. On crée donc un code spécial pour la base logement (`code_logement`) :
- si le code d'origine existe dans la base logement, on l'utilise ;
- sinon, on prend le code 2026.

| Cas | Code utilisé pour le recensement | Code utilisé pour LOVAC |
|---|---|---|
| Commune jamais modifiée | Code d'origine | Code d'origine |
| Fusion avant 2024 | Code 2026 | Code 2026 |
| Fusion en 2025 ou 2026 | Code d'origine | Code 2026 |

#### Étape 1 : ajout de la base logement (sur la commune)
- **Clé :** `code_logement` (table cible) ↔ `COM` (base logement).
- La base logement n'a qu'une année (2022) : **la même valeur est recopiée pour toutes les années** d'une commune, de 2014 à 2025.
- Pour les ventes de 2014 à 2021, on utilise donc un recensement **postérieur** à la vente (signalé par `rp_anticipe`). C'est acceptable car le parc de logements évolue lentement.

#### Étape 2 : ajout de LOVAC (sur la commune ET l'année)
- **Clé :** `code_2026` + `annee` (table cible) ↔ `COM` + `annee_vente_associee` (LOVAC).
- **Règle de date :** le millésime N de LOVAC décrit la situation au **1er janvier N-1**. Une vente de l'année N reçoit le millésime N : on n'utilise que de l'information antérieure à la vente.
  - *Exemple :* une vente de 2023 reçoit le millésime 2023, qui décrit les logements vacants au 1er janvier 2022.
- LOVAC commence au millésime 2020 : les ventes de **2014 à 2019 n'ont pas de données LOVAC** (valeurs vides).

#### Étape 3 : séparation entraînement / test (en mémoire)
- Entraînement : 2014-2024. **Seules ces années servent à choisir les variables.**
- Test : 2025, mis de côté et jamais regardé, pour ne pas biaiser le choix (fuite de données).

#### Contrôles effectués

| Contrôle | Résultat |
|---|---|
| Nombre de lignes après chaque merge | 367 779, identique à la table cible : aucune ligne perdue ni dupliquée |
| Une seule ligne par commune × année | Oui |
| Lignes sans données logement | 63 (négligeable) |
| Lignes avec LOVAC, 2014-2019 | 0 % (LOVAC n'existe pas encore) |
| Lignes avec LOVAC, 2020-2025 | 100 % (taux complété par l'EPCI si secret statistique) |

#### Limites du merge
- Une commune fusionnée reçoit les caractéristiques de **la nouvelle commune entière**. Par exemple, le village A reçoit les valeurs de C, qui regroupe A et B. Ses caractéristiques propres sont donc lissées. Ces lignes sont signalées par `commune_fusionnee`.
- Pour les rares communes **scindées** (une commune divisée en plusieurs), seul le premier code de la table de passage est conservé.
- Le même recensement 2022 est utilisé pour les 12 années : il ne capte pas l'évolution du parc entre 2014 et 2025.